In [ ]:
import random, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
import pandas as pd
import math
from collections import Counter
import pickle

from utilities_FunRec import (build_bridge_class, assign_fragility_by_bridge_class, build_component_quantity,
    sample_damage_correlated_baker,formalize_CountDamagedQty,map_comp_RC, 
    sample_order_IF, sample_replacementdur, sample_comp_repairdur,  assign_replacement_duration_bounds,
    order_comp_repairdur, decisiontree_reopeningFS, rd_num_byMean, sample_closedlanenum)

# Functional Recovery Analysis of Bridges

## 1. User Inputs
Specify all analysis options, bridge geometry, repair assumptions, and simulation settings.

<span style="color:yellow">Users are required to specify all input variables based on the design of the bridge case study of interest. Before proceeding to the subsequent analysis procedures, users are strongly encouraged to review all variables listed in the cell below.</span>


In [ ]:
# =============================================================================
# USER INPUTS
# =============================================================================

# -----------------------------------------------------------------------------
# Analysis settings
# -----------------------------------------------------------------------------
num_rlz = 10000          # Number of Monte Carlo realizations
IM_fixed = 0.45       # Sa(1s) intensity measure [g]

# Correlation weights for damage sampling:
# [1, 0, 0]       -> fully dependent
# [0, 0, 1]       -> independent
# [0.2, 0.6, 0.2] -> partially correlated, recommended
correlation_weights = [0, 0, 1]


# -----------------------------------------------------------------------------
# Bridge geometry and classification
# -----------------------------------------------------------------------------
bridge_inputs = {
    "design_era": "E3",          # E1: pre-1970; E2: 1970-1990; E3: post-1990
    "num_span": 5,
    "num_cols_per_bent": 1,      # Use "NA" for single-span bridges
    "column_shape": "C",         # C: circular; O: oblong; R: rectangular; "NA" for S1
    "height": 35,                # Bridge height [ft]
    "abutment_type": "D",        # S: seat-type; D: diaphragm-type
    "num_lanes_before": 4,       # Number of lanes per direction before earthquake
}
# Automatically reset unused inputs for single-span bridges
bridge_inputs.update({"num_cols_per_bent": "NA", "column_shape": "NA"} if bridge_inputs["num_span"] == 1 else {})

# -----------------------------------------------------------------------------
# Component quantities and materials
# -----------------------------------------------------------------------------
# Quantities refer to the number of independent repairable component sets.
CompQty = {
    # -------------------------------------------------------------------------
    # Superstructure / Substructure
    # -------------------------------------------------------------------------
    "Col": (bridge_inputs["num_span"] - 1) * bridge_inputs["num_cols_per_bent"],
    "ColFnd": (bridge_inputs["num_span"] - 1) * bridge_inputs["num_cols_per_bent"],
    "Super": bridge_inputs["num_span"],

    # -------------------------------------------------------------------------
    # Abutment-related components
    # (assume two abutments)
    # -------------------------------------------------------------------------
    "AbFnd": 2,
    "Backwall": 2,
    "ApproSlab": 2,

    # Seat-type abutment components
    # Set to zero for diaphragm abutments.
    "Seat_ab": 0,
    "Bearing_ab": 0,
    "Key_ab": 0,
    "JointSeal_ab": 0,

    # -------------------------------------------------------------------------
    # Interior support (bent / in-span hinge) components
    # Set to zero if the bridge has no interior hinges or expansion joints.
    # -------------------------------------------------------------------------
    "Seat_super": 0,
    "Bearing_super": 0,
    "Key_super": 0,
    "JointSeal_super": 0,
}

# Whether any interior-support components are present
has_super_location_components = any(
    CompQty[comp] > 0 for comp in ("Seat_super","Bearing_super", "Key_super", "JointSeal_super"))

# Material type only needs to be specified for components where non-concrete
# behavior changes repair duration or concrete curing time.
ColSuperMatType_dict = {
    "Col": "concrete",
    "Super": "concrete",
}


# -----------------------------------------------------------------------------
# Repair work scheme
# -----------------------------------------------------------------------------
# Number of workers assigned to each repair crew
WorkerAllo_percrew = {
    "Col": 4, "Seat_ab": 4, "Super": 5,
    "ColFnd": 4, "AbFnd": 4, "Backwall": 4,
    "Bearing_ab": 4, "Key_ab": 4, "ApproSlab": 5,
    "JointSeal_ab": 5, "Seat_super": 4, "Bearing_super": 3,
    "Key_super": 4, "JointSeal_super": 5}

# Number of crews assigned to each component type
NumCrew_percomp = {
    "Col": 1, "Seat_ab": 1, "Super": 1,
    "ColFnd": 1, "AbFnd": 1, "Backwall": 1,
    "Bearing_ab": 1, "Key_ab": 1, "ApproSlab": 1,
    "JointSeal_ab": 1, "Seat_super": 1, "Bearing_super": 1,
    "Key_super": 1, "JointSeal_super": 1}

WorkHour_repairable = 8     # Repair work hours per day; must be between 8 and 24


# -----------------------------------------------------------------------------
# Replacement work scheme
# -----------------------------------------------------------------------------
Worker_Replace = 20                    # Number of workers assigned to replacement
num_concrete_pour_replacement = 1       # Each pour adds 28 days of curing time
WorkHour_replacement = 8                # Replacement work hours per day


# -----------------------------------------------------------------------------
# Emergency response protocol
# -----------------------------------------------------------------------------
emergency_protocol_flag = False
# True  -> emergency protocol activated for damaged bridges
# False -> normal non-emergency response

# -----------------------------------------------------------------------------
# Duration uncertainty
# -----------------------------------------------------------------------------
dispersion_assigned = 0.30     # Lognormal dispersion for repair/replacement duration


## 2. Embedded Database

This section defines the built-in databases used by the recovery framework, including:

- Functionality state definitions
- Repair class mappings
- Impeding-factor distributions
- Replacement duration database
- Component repair duration database
- Worker allocation and curing-time database

<span style="color:yellow"> The parameters provided in the cell below are informed by interviews with California bridge practitioners and may therefore be most applicable to bridges in California. Their use for bridges outside of California should be undertaken with caution. Users are encouraged to modify these parameters based on local jurisdictional requirements and practices when conducting analyses for regions outside of California.</span>


In [53]:
# =============================================================================
# EMBEDDED DATABASE
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Fragility assignment
# -----------------------------------------------------------------------------
bridge_class = build_bridge_class(
    design_era=bridge_inputs["design_era"],
    num_span=bridge_inputs["num_span"],
    num_cols_per_bent=bridge_inputs["num_cols_per_bent"],
    column_shape=bridge_inputs["column_shape"],
    abutment_type=bridge_inputs["abutment_type"],
)


with open("FRA_LIB.pkl", "rb") as f:
    FRA_LIB = pickle.load(f)

CompFra_dict = assign_fragility_by_bridge_class(
    bridge_class=bridge_class,
    FRA_LIB=FRA_LIB,
    include_super_location_components=True,
)


# -----------------------------------------------------------------------------
# 1. Mapping from Repair Class (RC) to Functionality State (FS)
# -----------------------------------------------------------------------------
RC_to_FS = {
    1: (0, "Fully Repaired"),
    2: (1, "Fully Functional"),
    3: (2, "Moderate Lane Closure"),
    4: (3, "Extensive Lane Closure"),
    5: (7, "Complete Closure"),

    # Reopening-only functionality states.
    # The keys are artificial IDs and are not actual RC values.
    996: (4, "Partially Reopen with Weight Restriction"),
    997: (5, "Partially Reopen with Minor Lane Closure"),
    998: (6, "Partially Reopen with Weight Restrictions and Minor Lane Closure"),
}


# -----------------------------------------------------------------------------
# 2. Mapping from Repair Class (RC) to system damage state
# -----------------------------------------------------------------------------
RC_to_sysDS = {
    1: (0, "No Damage"),
    2: (1, "Slight Damage"),
    3: (2, "Moderate Damage"),
    4: (3, "Extensive Damage"),
    5: (4, "Complete Damage"),
}


# -----------------------------------------------------------------------------
# 3. Reopening functionality-state probabilities
# -----------------------------------------------------------------------------
# Returned reopening FS options are [4, 5, 6]:
# FS4: weight restriction
# FS5: minor lane closure
# FS6: weight restriction + minor lane closure
DecTreeProb_SuperAppro = [0.20, 0.10, 0.70]
DecTreeProb_AbutRelated = [0.60, 0.20, 0.20]


# -----------------------------------------------------------------------------
# 4. Impeding factor database [days]
# -----------------------------------------------------------------------------
# Format:
#   IF name: [
#       [EP not triggered, does not affect functionality],
#       [EP not triggered, affects functionality],
#       [EP triggered, bridge not complete],
#       [EP triggered, bridge complete],
#   ]
Impeding_dataset = {
    "IniInsp": [
        [0.02, 0.25],
        [0.02, 0.25],
        [0.02, 0.25],
        [0.02, 0.25],
    ],
    "InDepInsp": [
        [3, 7],
        [3, 7],
        [0.083, 1.5],
        [0, 0],
    ],
    "Financing": [
        [180, 720],
        [30, 60],
        [0, 0],
        [0, 0],
    ],
    "Design": [
        [0, 0],
        [30, 60],
        [7, 60],
        [14, 60],
    ],
    "Permitting": [
        [42, 90],
        [42, 90],
        [1, 7],
        [1, 7],
    ],
    "Contractor": [
        [360, 720],
        [90, 180],
        [0.25, 2],
        [0.25, 2],
    ],
}


# -----------------------------------------------------------------------------
# 5. Bridge replacement duration database
# -----------------------------------------------------------------------------
# Duration bounds do not include concrete curing time.
# Format:
#   [minimum duration, maximum duration] [days]
#
# Worker bounds:
#   [maximum workers, minimum workers]
#   maximum workers correspond to minimum duration
#   minimum workers correspond to maximum duration
num_span_for_database = bridge_inputs["num_span"]

RepDur_bridge_singlespan = [28, 58]
RepDur_bridge_twospan = [40, 86]
RepDur_bridge_multispan_l30 = [40, 86]
RepDur_bridge_multispan_g30_l100 = [
    40 + 30 * (num_span_for_database - 2),
    86 + 30 * (num_span_for_database - 2),
]
RepDur_bridge_multispan_g100 = [
    40 + 180 * (num_span_for_database - 2),
    86 + 180 * (num_span_for_database - 2),
]

RepDur_bridge_singlespan_WorkerBound = [20, 5]
RepDur_bridge_twospan_WorkerBound = [30, 10]
RepDur_bridge_multispan_l30_WorkerBound = [35, 20]
RepDur_bridge_multispan_g30_l100_WorkerBound = [
    35 + 15 * (num_span_for_database - 2),
    20 + 15 * (num_span_for_database - 2),
]
RepDur_bridge_multispan_g100_WorkerBound = [
    35 + 15 * (num_span_for_database - 2),
    20 + 15 * (num_span_for_database - 2),
]


# -----------------------------------------------------------------------------
# 6. Component repair duration database
# -----------------------------------------------------------------------------
# Format:
#   CompName: [
#       [min duration for DS0, max duration for DS0],
#       [min duration for DS1, max duration for DS1],
#       ...
#   ]
#
# Notes:
#   - Complete damage is not included here because replacement is handled separately.
#   - Component repair durations include concrete curing time.
RepDur_comp_dict = {
    "Col": [
        [1e-5, 1e-4],
        [3, 5],
        [3, 10],
        [34.5, 48],
    ],
    "Seat_ab": [
        [1e-5, 1e-4],
        [1e-5, 1e-4],
        [4, 11],
        [10, 23],
    ],
    "Seat_super": [
        [1e-5, 1e-4],
        [1e-5, 1e-4],
        [4, 11],
        [10, 23],
    ],
    "Super": [
        [1e-5, 1e-4],
        [3, 5],
        [3, 10],
    ],
    "ColFnd": [
        [1e-5, 1e-4],
        [1e-5, 1e-5],
        [6, 17],
    ],
    "AbFnd": [
        [1e-5, 1e-4],
        [1e-5, 1e-4],
        [4, 12],
    ],
    "Bearing_ab": [
        [1e-5, 1e-4],
        [1e-5, 1e-4],
        [2, 7],
    ],
    "Bearing_super": [
        [1e-5, 1e-4],
        [1e-5, 1e-4],
        [2, 7],
    ],
    "Key_ab": [
        [1e-5, 1e-4],
        [3, 5],
        [11.5, 22],
    ],
    "Key_super": [
        [1e-5, 1e-4],
        [3, 5],
        [11.5, 22],
    ],
    "Backwall": [
        [1e-5, 1e-4],
        [4, 9],
        [12.5, 26],
    ],
    "ApproSlab": [
        [1e-5, 1e-4],
        [1, 4],
        [2, 8],
    ],
    "JointSeal_ab": [
        [1e-5, 1e-4],
        [2, 4],
        [2, 4],
    ],
    "JointSeal_super": [
        [1e-5, 1e-4],
        [2, 4],
        [2, 4],
    ],
}


# -----------------------------------------------------------------------------
# 7. Component repair worker-bound database
# -----------------------------------------------------------------------------
# Format:
#   CompName: [
#       [max workers for DS0, min workers for DS0],
#       [max workers for DS1, min workers for DS1],
#       ...
#   ]
RepDur_WorkerBound_dict = {
    "Col": [
        [1, 1],
        [6, 1],
        [6, 2],
        [6, 3],
    ],
    "Seat_ab": [
        [1, 1],
        [6, 1],
        [6, 3],
        [6, 3],
    ],
    "Seat_super": [
        [1, 1],
        [6, 1],
        [6, 3],
        [6, 3],
    ],
    "Super": [
        [1, 1],
        [6, 4],
        [6, 4],
    ],
    "ColFnd": [
        [1, 1],
        [6, 3],
        [6, 3],
    ],
    "AbFnd": [
        [1, 1],
        [6, 3],
        [6, 3],
    ],
    "Bearing_ab": [
        [1, 1],
        [6, 2],
        [6, 2],
    ],
    "Bearing_super": [
        [1, 1],
        [6, 2],
        [6, 2],
    ],
    "Key_ab": [
        [1, 1],
        [6, 1],
        [6, 3],
    ],
    "Key_super": [
        [1, 1],
        [6, 1],
        [6, 3],
    ],
    "Backwall": [
        [1, 1],
        [6, 2],
        [6, 2],
    ],
    "ApproSlab": [
        [1, 1],
        [7, 4],
        [7, 4],
    ],
    "JointSeal_ab": [
        [1, 1],
        [6, 4],
        [6, 4],
    ],
    "JointSeal_super": [
        [1, 1],
        [6, 4],
        [6, 4],
    ],
}


# -----------------------------------------------------------------------------
# 8. Concrete curing-time database [days]
# -----------------------------------------------------------------------------
# Format:
#   CompName: [curing time for DS0, curing time for DS1, ...]
ConcreteCuringTime_comp_dict = {
    "Col": [0, 0, 0, 28],
    "Seat_ab": [0, 0, 0, 0],
    "Seat_super": [0, 0, 0, 0],
    "Super": [0, 0, 0],
    "ColFnd": [0, 0, 0],
    "AbFnd": [0, 0, 0],
    "Bearing_ab": [0, 0, 0],
    "Bearing_super": [0, 0, 0],
    "Key_ab": [0, 0, 7],
    "Key_super": [0, 0, 7],
    "Backwall": [0, 0, 7],
    "ApproSlab": [0, 0, 0],
    "JointSeal_ab": [0, 0, 0],
    "JointSeal_super": [0, 0, 0],
}

## 3. Sample Component Damage

**Objective**

Generate correlated component-level damage states for all Monte Carlo realizations.

**Inputs**
- Bridge fragility models
- Component quantities
- Correlation assumptions
- Ground-motion intensity measure

**Key Outputs**
- `DamageSample_CompModel_Qty`
- `DamageSample_Comp_Qty`

In [54]:
random.seed(1223)
np.random.seed(1223)

# -- Sample correlated damage
# Define a map from physical components to fragility models
Comp_to_CompModel = {
    "Col": ["Column"],
    "Seat_ab": ["Unseating"],
    "Super": ["DeckMax"],

    "ColFnd": ["FndRot", "FndTran"],
    "AbFnd": ["AbAct", "AbPass", "AbTran"],
    "Backwall": ["AbAct", "AbPass"],
    "ApproSlab": ["BackfillSettlement"],

    "Bearing_ab": ["Bearing"],
    "Key_ab": ["Key"],
    "JointSeal_ab": ["Seal"],

    "Seat_super": ["Seat_super"],
    "Bearing_super": ["Bearing_super"],
    "Key_super": ["Key_super"],
    "JointSeal_super": ["JointSeal_super"]}

CompModelQty = {
    comp_model: 0
    for comp_models in Comp_to_CompModel.values()
    for comp_model in comp_models}

for comp_name, qty in CompQty.items():
    for comp_model in Comp_to_CompModel.get(comp_name, []):
        if comp_model in CompFra_dict["CompFra"]:
            CompModelQty[comp_model] = max(CompModelQty[comp_model], qty)

CompModelQty = {
    k: v for k, v in CompModelQty.items()
    if v > 0 and k in CompFra_dict["CompFra"]}
CompModelName_List = list(CompModelQty.keys())


# Determine what CompModelNames are regarded to be in a same group
# key: sys, value: subsys
IntraGroupRule = {
    "Substructure": ["Column", "FndRot", "FndTran", "Seat_super", "Bearing_super", "Key_super", "JointSeal_super"],
    "Abutment": ["Unseating", "Seal", "AbAct", "AbPass", "AbTran", "BackfillSettlement", "Key", "Bearing"],
    "Superstructure": ["DeckMax"],}

IntraGroupRule = {
    group: [m for m in models if m in CompModelQty]
    for group, models in IntraGroupRule.items()}
# remove empty groups before sampling
IntraGroupRule = {
    group: models
    for group, models in IntraGroupRule.items() if len(models) > 0}

# - DamageSample_CompModel_Qty - row num = comp num; col num = rlz num
DamageSample_CompModel_Qty = sample_damage_correlated_baker(
    IM_fixed, CompModelName_List, CompModelQty,
    IntraGroupRule, CompFra_dict, correlation_weights, num_rlz)

# - Map CompModel damage back to physical components
DamageSample_Comp_Qty = {}

for comp_name, qty in CompQty.items():
    if qty == 0:
        continue
    comp_models = [
        m for m in Comp_to_CompModel.get(comp_name, [])
        if m in DamageSample_CompModel_Qty]

    if len(comp_models) == 0:
        continue

    ds_arrays = [
        np.array(DamageSample_CompModel_Qty[m][:qty])
        for m in comp_models]

    DamageSample_Comp_Qty[comp_name] = np.max(
        np.stack(ds_arrays, axis=0),
        axis=0).tolist()

In [55]:
# Get the porportion of damage count per comp
Percent_Damage_CompModel_Qty = {key:None for key in DamageSample_CompModel_Qty.keys()}
Percent_Damage_Comp_Qty = {key:None for key in DamageSample_Comp_Qty.keys()}

for CompModelName in Percent_Damage_CompModel_Qty.keys():
    DamageSample_current =  DamageSample_CompModel_Qty[CompModelName]
    num_comp = len(DamageSample_current)
    possible_DS = 4 if CompModelName.lower() in ['col', 'seat'] else 2
    DSSample_count = [ [None]*possible_DS for _ in range(num_comp)]
    for compnum_idx in range(num_comp):
        DS_count_compnum = Counter(DamageSample_current[compnum_idx])
        DS_freq_compnum = [DS_count_compnum.get(ds,0)/num_rlz for ds in range(possible_DS+1)]
        DSSample_count[compnum_idx] = DS_freq_compnum
    Percent_Damage_CompModel_Qty[CompModelName] = DSSample_count
    
for CompName in Percent_Damage_Comp_Qty.keys():
    DamageSample_current =  DamageSample_Comp_Qty[CompName]
    num_comp = len(DamageSample_current)
    possible_DS = 4 if (CompName == "Col" or "seat" in CompName.lower()) else 2
    DSSample_count = [ [None]*possible_DS for _ in range(num_comp)]
    for compnum_idx in range(num_comp):
        DS_count_compnum = Counter(DamageSample_current[compnum_idx])
        DS_freq_compnum = [DS_count_compnum.get(ds,0)/num_rlz for ds in range(possible_DS+1)]
        DSSample_count[compnum_idx] = DS_freq_compnum
    Percent_Damage_Comp_Qty[CompName] = DSSample_count

In [56]:
# only retrieve the entries with non-empty (meaning, non-zero comp quantity) in the list
NonEmptyCompModelName_List = [key for key, value in DamageSample_CompModel_Qty.items() if value]
NonEmptyCompName_List = [key for key, value in DamageSample_Comp_Qty.items() if value]

DamageSample_CompModel_Qty = {key: DamageSample_CompModel_Qty[key] for key in NonEmptyCompModelName_List} # not needed further
Percent_Damage_Comp_Qty = {key: Percent_Damage_Comp_Qty[key] for key in NonEmptyCompName_List}
DamageSample_Comp_Qty = {key: DamageSample_Comp_Qty[key] for key in NonEmptyCompName_List}

## 4. Convert Damage States to Repair Classes

**Function:** `map_comp_RC()`

Convert sampled component damage states into component repair classes while accounting for damage quantity.

In [57]:
##---- Convert 'DamageSample_Comp_Qty' (sampled scattered DS per qty, per rlz) into a count dict 'CountDamagedQty' (per rlz, per DS )
CountDamagedQty = formalize_CountDamagedQty(NonEmptyCompName_List,DamageSample_Comp_Qty)
# row: num_rlz; col: DS. Value in (idx_row, idx_col) indicate how many components have DS {idx_col} in rlz {idx_row}

RepairClass_dict = {key: None for key in NonEmptyCompName_List}  # each key having a list with length = num_rlz
##---- Get the RC for each comp per rlz
for CompName_this in NonEmptyCompName_List:
    RepairClass_dict[CompName_this] = map_comp_RC(CountDamagedQty[CompName_this], CompName_this)

## 5. Determine Initial System Functionality

Determine the initial system damage state (`sysDS`) and initial functionality state (`IFS`) immediately after the earthquake.

In [58]:
FS_rlz= []
sysDS_rlz = []

for rlz_idx in range(num_rlz):
    # Get the max RC for each realization across all components
    maxRC_perrlz = max(RC_perComp[rlz_idx] for RC_perComp in RepairClass_dict.values())
    # Assign FS and sysDS based on the worst-case RC
    FS_rlz.append(RC_to_FS[maxRC_perrlz][0])
    sysDS_rlz.append(RC_to_sysDS[maxRC_perrlz][0])
    
#print(FS_rlz)
#plt.hist(sysDS_rlz)

## 6. Sample Initial Lane Closures

Sample the number of closed lanes associated with the initial functionality state.

In [59]:
random.seed(1223)
np.random.seed(1223)

ClosedLaneNum_Initial = []
ClosedLaneNum_Initial = [sample_closedlanenum('Initial', FS_scalar, lane_before = bridge_inputs["num_lanes_before"]) for FS_scalar in FS_rlz]

## 7. Sample Impeding Factors

**Function:** `sample_order_IF()`

Sample all impeding factors and determine the governing delay before repair or replacement can begin.

In [60]:
random.seed(1223)
np.random.seed(1223)

if emergency_protocol_flag:
    # Emergency protocol applied to all damaged bridges
    emergency_protocol_flag_list = [1] * len(sysDS_rlz)
else:
    # No emergency protocol
    emergency_protocol_flag_list = [0] * len(sysDS_rlz)

IF_sampled_list, IF_sum_list = sample_order_IF(sysDS_rlz,Impeding_dataset,emergency_protocol_flag_list)

## 8. Bridge Replacement Duration

Sample bridge replacement duration for realizations classified as complete damage (`sysDS = 4`).

In [61]:
repldur_min, repldur_max, replworker_max, replworker_min = assign_replacement_duration_bounds(
    num_span=bridge_inputs["num_span"],
    height=bridge_inputs["height"],
    RepDur_bridge_singlespan=RepDur_bridge_singlespan,
    RepDur_bridge_singlespan_WorkerBound=RepDur_bridge_singlespan_WorkerBound,
    RepDur_bridge_twospan=RepDur_bridge_twospan,
    RepDur_bridge_twospan_WorkerBound=RepDur_bridge_twospan_WorkerBound,
    RepDur_bridge_multispan_l30=RepDur_bridge_multispan_l30,
    RepDur_bridge_multispan_l30_WorkerBound=RepDur_bridge_multispan_l30_WorkerBound,
    RepDur_bridge_multispan_g30_l100=RepDur_bridge_multispan_g30_l100,
    RepDur_bridge_multispan_g30_l100_WorkerBound=RepDur_bridge_multispan_g30_l100_WorkerBound,
    RepDur_bridge_multispan_g100=RepDur_bridge_multispan_g100,
    RepDur_bridge_multispan_g100_WorkerBound=RepDur_bridge_multispan_g100_WorkerBound,
)

## 9. Component Repair Duration

Sample repair duration for each damaged component and compute the governing bridge repair duration.

In [62]:
RepDur_sum_rlz = [None] * num_rlz
RepDur_sampled_comp_rlz = [None] * num_rlz

for rlz_idx, sysDS in enumerate(sysDS_rlz):

    # -------------------------------------------------------------------------
    # Case 1: complete damage -> bridge replacement
    # -------------------------------------------------------------------------
    if sysDS == 4:
        RepDur_sampled_comp_rlz[rlz_idx] = "Complete"

        RepDur_sum_rlz[rlz_idx] = sample_replacementdur(
            Worker_Replace_input=Worker_Replace,
            repldur_min_input=repldur_min,
            repldur_max_input=repldur_max,
            replworker_max_input=replworker_max,
            replworker_min_input=replworker_min,
            WorkHour_input=WorkHour_replacement,
            num_concrete_pour_input=num_concrete_pour_replacement,
            dispersion_assigned_scalar=dispersion_assigned,
        )

    # -------------------------------------------------------------------------
    # Case 2: repairable damage -> component repair
    # -------------------------------------------------------------------------
    else:
        # Extract component damage states for the current realization
        DamageSample_Comp_Qty_rlz = {
            comp_name: [
                ds_per_qty[rlz_idx]
                for ds_per_qty in damage_samples_per_qty
            ]
            for comp_name, damage_samples_per_qty in DamageSample_Comp_Qty.items()
        }

        # Sample component-level repair duration
        RepDur_sampled_comp_rlz[rlz_idx] = sample_comp_repairdur(
            DS_comp_rlz=DamageSample_Comp_Qty_rlz,
            RepDur_comp_dict_input=RepDur_comp_dict,
            RepDur_WorkerBound_dict_input=RepDur_WorkerBound_dict,
            WorkerAllo_Comp_input=WorkerAllo_percrew,
            NumCrew_percomp_input=NumCrew_percomp,
            WorkHour_input=WorkHour_repairable,
            ConcreteCuringTime_comp_dict_input=ConcreteCuringTime_comp_dict,
            ColSuperMatType_input=ColSuperMatType_dict,
            dispersion_assigned_scalar=dispersion_assigned,
        )

        # Order component repairs into system-level repair duration
        RepDur_sum_rlz[rlz_idx] = order_comp_repairdur(
            RepDur_sampled_dict=RepDur_sampled_comp_rlz[rlz_idx],
            CompName_List_input=CompQty,
        )

## 10. Reopening Functionality

Determine the reopening functionality state, reopening lane closures, and total recovery time after accounting for repairs and impeding factors.

In [63]:
##---- Sampling reopening FS
FS_rlz_Reopening=[]
ReopeningTriggeringFlag_rlz = []

for rlz_idx in range(num_rlz):
    RC_comp_thisrlz  = {CompName: RClist[rlz_idx] for CompName, RClist in RepairClass_dict.items()}
    RepDur_sampled_thisrlz = RepDur_sampled_comp_rlz[rlz_idx]
    FS_rlz_Reopening_thisrlz,ReopeningTriggeringFlag_thisrlz =\
                        decisiontree_reopeningFS(RC_comp_thisrlz,
                        RepDur_sampled_thisrlz, FS_rlz[rlz_idx],
                        DecTreeProb_SuperAppro, DecTreeProb_AbutRelated)
    FS_rlz_Reopening.append(FS_rlz_Reopening_thisrlz)
    ReopeningTriggeringFlag_rlz.append(ReopeningTriggeringFlag_thisrlz)

In [64]:
##---- Closed lane # sampling under reopening stage
ClosedLaneNum_Reopening = []
weight_restriction_tag_Reopening = []
for rlz_idx in  range(num_rlz):
    closedlane_reopening_thisrlz, weight_restriction_tag_thisrlz = sample_closedlanenum(
        "Reopening", FS_rlz_scalar = FS_rlz_Reopening[rlz_idx], 
        lane_before = bridge_inputs["num_lanes_before"], closed_lane_IFS_rlz_scalar =  ClosedLaneNum_Initial[rlz_idx])
    
    ClosedLaneNum_Reopening.append(closedlane_reopening_thisrlz)
    weight_restriction_tag_Reopening.append(weight_restriction_tag_thisrlz)
    
#ClosedLaneNum_Reopening = [sample_closedlanenum('Reopening', FS_rlz_scalar = FS_rlz_Reopening[rlz_idx], lane_before = num_lanes_before, closed_lane_IFS_rlz_scalar =  ClosedLaneNum_Initial[rlz_idx]) for rlz_idx in  range(num_rlz)]

## 11. Save Results

Save key simulation outputs to the specified local directory.

In [65]:
data= { #change  fullydepen, indep, partiallydepen
    'DamageSample_Comp_Qty': DamageSample_Comp_Qty,
    'FS_rlz': FS_rlz,
    'sysDS_rlz': sysDS_rlz,
    'RepairClass_dict': RepairClass_dict,
    'ClosedLaneNum_Initial': ClosedLaneNum_Initial,
    'FS_rlz_Reopening': FS_rlz_Reopening,
    'ClosedLaneNum_Reopening': ClosedLaneNum_Reopening,
    'IF_sampled_list': IF_sampled_list,
    'IF_sum_list': IF_sum_list,
    'RepDur_sum_rlz':RepDur_sum_rlz ,
    'RepDur_sampled_comp_rlz':RepDur_sampled_comp_rlz
}

import pickle 

with open('Results.pkl', 'wb') as file: # change file name
    pickle.dump((data), file) # change variable name